In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.2f}".format)

BASE_DIR = Path("../data")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
orders = pd.read_csv(RAW_DIR / "orders_v2_full.csv")
order_items = pd.read_csv(RAW_DIR / "order_items_v2_full.csv")

print("Orders shape:", orders.shape)
print("Order Items shape:", order_items.shape)

Orders shape: (200000, 21)
Order Items shape: (430612, 11)


In [3]:
print("Orders columns:")
print(orders.columns)

print("\nOrder items columns:")
print(order_items.columns)

Orders columns:
Index(['order_id', 'user_id', 'city', 'tier', 'restaurant_id', 'cuisine',
       'zone_type', 'distance_km', 'order_date', 'season', 'user_age',
       'is_member', 'delivery_fee', 'subtotal', 'total_order_value',
       'avg_user_value', 'cart_size', 'has_main', 'has_side', 'has_drink',
       'has_dessert'],
      dtype='str')

Order items columns:
Index(['order_id', 'item_id', 'item_name', 'category', 'price',
       'restaurant_id', 'cuisine', 'is_vegetarian', 'spice_level',
       'price_tier', 'addon_rate'],
      dtype='str')


In [4]:
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

In [5]:
print(orders["order_date"].dtype)
print(orders["order_date"].isna().sum())

datetime64[us]
0


In [6]:
# Unique order_id check
duplicate_mask = orders["order_id"].duplicated(keep=False)
if duplicate_mask.any():
    duplicate_count = int(duplicate_mask.sum())
    duplicate_ids = int(orders.loc[duplicate_mask, "order_id"].nunique())
    print(f"Found {duplicate_count} duplicate rows across {duplicate_ids} order_id values.")
    print("Deduplicating orders by order_id (keeping first row).")
    orders = orders.drop_duplicates(subset=["order_id"], keep="first").copy()

assert orders["order_id"].is_unique, "Duplicate order_id found after deduplication!"

# Orphan items check
orphan_orders = set(order_items["order_id"]) - set(orders["order_id"])
assert len(orphan_orders) == 0, "Orphan order_items detected!"

print("Data integrity checks passed ")


Found 39825 duplicate rows across 19189 order_id values.
Deduplicating orders by order_id (keeping first row).
Data integrity checks passed 


In [7]:
item_totals = (
    order_items.groupby("order_id")["price"].sum().reset_index()
)

item_totals = item_totals.rename(columns={"price": "price_calc"})

merged_check = orders.merge(item_totals, on="order_id", how="left")

diff = (merged_check["subtotal"] - merged_check["price_calc"]).abs().mean()
print("Average subtotal diff:", diff)

Average subtotal diff: 79.73796302491024


In [8]:
cart_features = (
    order_items
    .groupby("order_id")
    .agg(
        cart_size=("item_id", "count"),
        cart_total=("price", "sum"),
        avg_item_price=("price", "mean"),
        has_main=("category", lambda x: int("main" in x.values)),
        has_side=("category", lambda x: int("side" in x.values)),
        has_drink=("category", lambda x: int("drink" in x.values)),
        has_dessert=("category", lambda x: int("dessert" in x.values))
    )
    .reset_index()
)

orders = orders.merge(cart_features, on="order_id", how="left")

print("Cart features added ")

Cart features added 


In [9]:
orders["hour"] = orders["order_date"].dt.hour
orders["day_of_week"] = orders["order_date"].dt.dayofweek
orders["month"] = orders["order_date"].dt.month
orders["is_weekend"] = orders["day_of_week"].isin([5, 6]).astype(int)

def map_time_bucket(h):
    if 7 <= h <= 10:
        return "breakfast"
    elif 12 <= h <= 15:
        return "lunch"
    elif 19 <= h <= 22:
        return "dinner"
    else:
        return "other"

orders["time_bucket"] = orders["hour"].apply(map_time_bucket)

print("Temporal features added ")

Temporal features added 


In [10]:
user_features = (
    orders
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "count"),
        avg_order_value=("subtotal", "mean"),
        max_order_value=("subtotal", "max"),
        first_order=("order_date", "min"),
        last_order=("order_date", "max")
    )
    .reset_index()
)

latest_time = orders["order_date"].max()

user_features["days_since_last_order"] = (
    latest_time - user_features["last_order"]
).dt.days

user_features["user_tenure_days"] = (
    latest_time - user_features["first_order"]
).dt.days

print("User features created ")

User features created 


In [11]:
user_features.to_csv(PROCESSED_DIR / "user_features.csv", index=False)

In [12]:
item_features = (
    order_items
    .groupby("item_id")
    .agg(
        item_order_count=("order_id", "nunique"),
        avg_price=("price", "mean")
    )
    .reset_index()
)

item_features["popularity_rank"] = (
    item_features["item_order_count"]
    .rank(method="dense", ascending=False)
)

print("Item features created ")

Item features created 


In [13]:
item_features.to_csv(PROCESSED_DIR / "item_features.csv", index=False)

In [14]:
basket = (
    order_items
    .groupby("order_id")["item_id"]
    .apply(list)
)

pair_counter = Counter()

for items in basket:
    unique_items = list(set(items))
    for pair in combinations(unique_items, 2):
        pair_counter[tuple(sorted(pair))] += 1

cooccurrence = pd.DataFrame(
    [(i, j, c) for (i, j), c in pair_counter.items()],
    columns=["item_i", "item_j", "count"]
)

print("Co-occurrence matrix created ")


Co-occurrence matrix created 


In [15]:
cooccurrence.to_pickle(PROCESSED_DIR / "cooccurrence.pkl")

In [16]:
orders.to_csv(PROCESSED_DIR / "orders_enriched.csv", index=False)

print("All preprocessing complete ")

All preprocessing complete 
